
<a href="https://colab.research.google.com/github/USERNAME/00631L-quant/blob/main/notebooks/00631L_Colab_V3.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# 00631L Quant V3

手機版一鍵執行：下載資料、建立資料庫、計算指標、批次回測、策略排名、每日決策。


In [ ]:

#@title ① 安裝套件並連接 Google Drive
!pip -q install yfinance pandas numpy pyarrow

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE = Path('/content/drive/MyDrive/00631L_Quant_V3')
BASE.mkdir(parents=True, exist_ok=True)
print("✅ Google Drive 資料夾：", BASE)


In [ ]:

#@title ② 下載市場資料
import sqlite3, json, math
import pandas as pd
import numpy as np
import yfinance as yf
from itertools import product
from dataclasses import dataclass, asdict

SYMBOLS = {
    "etf": "00631L.TW",
    "twii": "^TWII",
    "nasdaq": "^IXIC",
    "sox": "^SOX",
    "sp500": "^GSPC",
    "vix": "^VIX",
    "dxy": "DX-Y.NYB",
    "us10y": "^TNX",
    "tsm_adr": "TSM",
}

def download_one(symbol):
    df = yf.download(symbol, start="2014-10-31", auto_adjust=False,
                     progress=False, threads=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = [c[0] for c in df.columns]
    if df.empty:
        raise RuntimeError(f"{symbol} 無資料")
    if "Adj Close" not in df.columns:
        df["Adj Close"] = df["Close"]
    if "Volume" not in df.columns:
        df["Volume"] = 0
    df.index = pd.to_datetime(df.index).tz_localize(None)
    return df[["Open","High","Low","Close","Adj Close","Volume"]].dropna(subset=["Close"])

data, errors = {}, {}
for name, symbol in SYMBOLS.items():
    try:
        data[name] = download_one(symbol)
        print(f"✅ {name}: {len(data[name])} 筆")
    except Exception as e:
        errors[name] = str(e)
        print(f"⚠️ {name}: {e}")

if "etf" not in data:
    raise RuntimeError("00631L 核心資料下載失敗")


In [ ]:

#@title ③ 建立 SQLite 資料庫
db_path = BASE / "market.db"

with sqlite3.connect(db_path) as con:
    con.execute('''
    CREATE TABLE IF NOT EXISTS prices(
        date TEXT NOT NULL,
        name TEXT NOT NULL,
        symbol TEXT NOT NULL,
        open REAL, high REAL, low REAL, close REAL, adj_close REAL, volume REAL,
        PRIMARY KEY(date, name)
    )
    ''')
    for name, x in data.items():
        rows = x.reset_index().rename(columns={
            "Date":"date","Open":"open","High":"high","Low":"low",
            "Close":"close","Adj Close":"adj_close","Volume":"volume"
        })
        rows["date"] = pd.to_datetime(rows["date"]).dt.strftime("%Y-%m-%d")
        rows["name"] = name
        rows["symbol"] = SYMBOLS[name]
        cols = ["date","name","symbol","open","high","low","close","adj_close","volume"]
        con.executemany(
            "INSERT OR REPLACE INTO prices VALUES (?,?,?,?,?,?,?,?,?)",
            rows[cols].itertuples(index=False, name=None)
        )

print("✅ SQLite 完成：", db_path)


In [ ]:

#@title ④ 計算技術指標與市場特徵
def rsi(close, n=14):
    d = close.diff()
    up = d.clip(lower=0).ewm(alpha=1/n, adjust=False).mean()
    dn = (-d.clip(upper=0)).ewm(alpha=1/n, adjust=False).mean()
    rs = up / dn.replace(0, np.nan)
    return 100 - 100/(1+rs)

def stochastic_kd(high, low, close, n=9):
    ll = low.rolling(n).min()
    hh = high.rolling(n).max()
    rsv = 100*(close-ll)/(hh-ll).replace(0,np.nan)
    k = rsv.ewm(alpha=1/3, adjust=False).mean()
    d = k.ewm(alpha=1/3, adjust=False).mean()
    return k, d

df = data["etf"].rename(columns=str.lower).copy()

for n in [5,10,20,60,120,240]:
    df[f"ma{n}"] = df["close"].rolling(n).mean()

df["rsi14"] = rsi(df["close"])
df["k"], df["d"] = stochastic_kd(df["high"],df["low"],df["close"])

fast = df["close"].ewm(span=12,adjust=False).mean()
slow = df["close"].ewm(span=26,adjust=False).mean()
df["dif"] = fast-slow
df["dea"] = df["dif"].ewm(span=9,adjust=False).mean()
df["macd_hist"] = df["dif"]-df["dea"]

prev = df["close"].shift(1)
tr = pd.concat([
    df["high"]-df["low"],
    (df["high"]-prev).abs(),
    (df["low"]-prev).abs()
], axis=1).max(axis=1)
df["atr14"] = tr.ewm(alpha=1/14, adjust=False).mean()

mid = df["close"].rolling(20).mean()
std = df["close"].rolling(20).std()
df["bb_mid"], df["bb_up"], df["bb_low"] = mid, mid+2*std, mid-2*std

weekly = df[["high","low","close"]].resample("W-FRI").agg(
    {"high":"max","low":"min","close":"last"}
)
wk, wd = stochastic_kd(weekly["high"],weekly["low"],weekly["close"])
df["week_k"] = wk.reindex(df.index, method="ffill")
df["week_d"] = wd.reindex(df.index, method="ffill")

for name, x in data.items():
    if name == "etf":
        continue
    y = x.copy()
    y[f"{name}_ma20"] = y["Adj Close"].rolling(20).mean()
    y[f"{name}_above_ma20"] = y["Adj Close"] > y[f"{name}_ma20"]
    y[f"{name}_ret1"] = y["Adj Close"].pct_change()
    df = df.join(y[[f"{name}_above_ma20", f"{name}_ret1"]], how="left")

ext_cols = [c for c in df.columns if "_above_ma20" in c or "_ret1" in c]
df[ext_cols] = df[ext_cols].ffill()
usable = df.dropna(subset=["ma240","week_k","rsi14"]).copy()
usable.to_parquet(BASE/"features.parquet")

print("✅ 特徵資料完成：", len(usable), "筆")


In [ ]:

#@title ⑤ 自動產生並回測大量策略
@dataclass(frozen=True)
class Strategy:
    name: str
    week_k_max: int
    rsi_max: int
    require_k_cross: bool
    require_macd_improve: bool
    require_close_ma20: bool
    require_twii_ma20: bool
    stop_loss: float
    take_profit: float
    max_hold: int

def make_strategies():
    out=[]; i=1
    for vals in product(
        [20,30,40],
        [30,35,40,45],
        [False,True],
        [False,True],
        [False,True],
        [False,True],
        [0.05,0.07,0.09],
        [0.10,0.15,0.20],
        [10,20,30,40]
    ):
        wk,rsi_,kx,macd,ma20,twii,sl,tp,mh=vals
        if tp <= sl:
            continue
        out.append(Strategy(f"S{i:05d}",wk,rsi_,kx,macd,ma20,twii,sl,tp,mh))
        i += 1
    return out

def signal_for(x,s):
    sig=(x["week_k"]<s.week_k_max)&(x["rsi14"]<s.rsi_max)
    if s.require_k_cross:
        sig &= (x["k"]>x["d"])&(x["k"].shift(1)<=x["d"].shift(1))
    if s.require_macd_improve:
        sig &= x["macd_hist"]>x["macd_hist"].shift(1)
    if s.require_close_ma20:
        sig &= x["close"]>x["ma20"]
    if s.require_twii_ma20 and "twii_above_ma20" in x:
        sig &= x["twii_above_ma20"].fillna(False)
    return sig.fillna(False)

commission_rate=0.001425
commission_discount=0.28
sell_tax=0.001
slippage=0.0005
round_trip_cost = (
    commission_rate*commission_discount + slippage +
    commission_rate*commission_discount + sell_tax + slippage
)

def backtest(x,s):
    sig=signal_for(x,s)
    trades=[]; i=1
    while i<len(x):
        if not bool(sig.iloc[i-1]):
            i += 1
            continue
        entry_i=i
        entry=float(x["open"].iloc[i])
        exit_i=min(i+s.max_hold,len(x)-1)
        exit_price=float(x["close"].iloc[exit_i])
        reason="max_hold"

        for j in range(i,min(i+s.max_hold,len(x)-1)+1):
            if float(x["low"].iloc[j]) <= entry*(1-s.stop_loss):
                exit_i=j; exit_price=entry*(1-s.stop_loss); reason="stop_loss"; break
            if float(x["high"].iloc[j]) >= entry*(1+s.take_profit):
                exit_i=j; exit_price=entry*(1+s.take_profit); reason="take_profit"; break

        net=exit_price/entry-1-round_trip_cost
        trades.append({
            "entry_date":x.index[entry_i],
            "exit_date":x.index[exit_i],
            "entry":entry,
            "exit":exit_price,
            "net_return":net,
            "hold_days":exit_i-entry_i+1,
            "reason":reason
        })
        i=exit_i+1

    t=pd.DataFrame(trades)
    if t.empty:
        return None,t

    r=t["net_return"]
    eq=(1+r).cumprod()
    dd=eq/eq.cummax()-1
    wins=r[r>0].sum()
    losses=-r[r<0].sum()
    pf=wins/losses if losses>0 else np.inf
    years=max((t["exit_date"].iloc[-1]-t["entry_date"].iloc[0]).days/365.25,1/365.25)

    return {
        "trades":len(t),
        "win_rate":(r>0).mean(),
        "avg_return":r.mean(),
        "profit_factor":pf,
        "max_drawdown":dd.min(),
        "cagr":eq.iloc[-1]**(1/years)-1,
        "total_return":eq.iloc[-1]-1
    }, t

split=int(len(usable)*0.70)
train,test=usable.iloc[:split],usable.iloc[split:]
rows=[]; trade_map={}

for s in make_strategies():
    mt,_=backtest(train,s)
    ms,tr=backtest(test,s)
    if mt is None or ms is None:
        continue
    if mt["trades"]+ms["trades"]<8 or ms["trades"]<3:
        continue
    if not np.isfinite(mt["profit_factor"]) or not np.isfinite(ms["profit_factor"]):
        continue

    row=asdict(s)
    for k,v in mt.items(): row[f"{k}_train"]=v
    for k,v in ms.items(): row[f"{k}_test"]=v

    row["score"]=(
        2.2*min(mt["profit_factor"],ms["profit_factor"]) +
        1.5*min(ms["profit_factor"],4) +
        2*ms["cagr"] +
        2*ms["max_drawdown"] +
        0.02*min(ms["trades"],30)
    )
    rows.append(row)
    trade_map[s.name]=tr

leaderboard=pd.DataFrame(rows).sort_values(
    ["score","profit_factor_test","max_drawdown_test"],
    ascending=[False,False,False]
).head(30)

if leaderboard.empty:
    raise RuntimeError("沒有策略通過最低樣本門檻")

leaderboard.to_csv(BASE/"strategy_leaderboard.csv",index=False)
print("✅ 回測完成")
display(leaderboard.head(10)[[
    "name","score","trades_test","win_rate_test",
    "profit_factor_test","avg_return_test",
    "max_drawdown_test","cagr_test"
]])


In [ ]:

#@title ⑥ 產生每日決策與手機版報告
from IPython.display import display, HTML

best=leaderboard.iloc[0]
best_s=Strategy(
    best["name"],int(best["week_k_max"]),int(best["rsi_max"]),
    bool(best["require_k_cross"]),bool(best["require_macd_improve"]),
    bool(best["require_close_ma20"]),bool(best["require_twii_ma20"]),
    float(best["stop_loss"]),float(best["take_profit"]),int(best["max_hold"])
)

latest_sig=bool(signal_for(usable,best_s).iloc[-1])
last=usable.iloc[-1]
position=20 if latest_sig else 0
status="🟢 開始布局" if latest_sig else "🟡 觀察"
action="下一交易日開盤建立20%部位" if latest_sig else "不進場，維持0%"

conditions={
    "周KD低於門檻": bool(last["week_k"]<best_s.week_k_max),
    "RSI低於門檻": bool(last["rsi14"]<best_s.rsi_max),
    "日KD黃金交叉": bool(last["k"]>last["d"] and usable["k"].iloc[-2]<=usable["d"].iloc[-2]),
    "MACD改善": bool(last["macd_hist"]>usable["macd_hist"].iloc[-2]),
    "收復20日均線": bool(last["close"]>last["ma20"]),
    "大盤站上20日均線": bool(last.get("twii_above_ma20",False))
}

decision={
    "data_date":str(usable.index[-1].date()),
    "strategy":best_s.name,
    "status":status,
    "action":action,
    "suggested_position_pct":position,
    "reference_close":float(last["close"]),
    "support_reference":float(last["bb_low"]),
    "resistance_reference":float(last["bb_up"]),
    "stop_loss_pct":best_s.stop_loss,
    "take_profit_pct":best_s.take_profit,
    "max_hold_days":best_s.max_hold,
    "conditions":conditions,
    "out_of_sample":{
        "trades":int(best["trades_test"]),
        "win_rate":float(best["win_rate_test"]),
        "profit_factor":float(best["profit_factor_test"]),
        "avg_return":float(best["avg_return_test"]),
        "max_drawdown":float(best["max_drawdown_test"]),
        "cagr":float(best["cagr_test"])
    }
}

(BASE/"daily_decision.json").write_text(
    json.dumps(decision,ensure_ascii=False,indent=2),encoding="utf-8"
)

checks="".join(
    f"<div style='padding:7px 0;border-bottom:1px solid #eee'>{'✅' if ok else '⬜'} {name}</div>"
    for name,ok in conditions.items()
)

html=f'''
<div style="max-width:520px;margin:auto;padding:18px;border-radius:18px;
box-shadow:0 4px 18px rgba(0,0,0,.12);
font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;
line-height:1.5;background:white;color:#111">
<h2 style="margin:0">00631L 每日決策</h2>
<div style="color:#666">資料截止：{decision["data_date"]}</div>
<hr>
<h3>{status}</h3>
<div style="font-size:34px;font-weight:700">{position}%</div>
<div style="color:#666">建議持股比例</div>
<hr>
<div><b>執行：</b>{action}</div>
<div><b>最佳策略：</b>{best_s.name}</div>
<div><b>參考收盤：</b>{last["close"]:.2f}</div>
<div><b>支撐參考：</b>{last["bb_low"]:.2f}</div>
<div><b>壓力參考：</b>{last["bb_up"]:.2f}</div>
<div><b>停損：</b>{best_s.stop_loss:.0%}</div>
<div><b>停利：</b>{best_s.take_profit:.0%}</div>
<div><b>最長持有：</b>{best_s.max_hold}日</div>
<hr>
<h3>條件檢查</h3>
{checks}
<hr>
<h3>樣本外績效</h3>
<div>交易次數：{int(best["trades_test"])}</div>
<div>勝率：{best["win_rate_test"]:.1%}</div>
<div>Profit Factor：{best["profit_factor_test"]:.2f}</div>
<div>平均單筆淨報酬：{best["avg_return_test"]:.2%}</div>
<div>最大回撤：{best["max_drawdown_test"]:.2%}</div>
<div>CAGR：{best["cagr_test"]:.2%}</div>
<div style="margin-top:14px;color:#777;font-size:13px">
歷史回測不保證未來報酬。
</div>
</div>
'''

(BASE/"daily_report.html").write_text(html,encoding="utf-8")
display(HTML(html))
print("✅ 已儲存到：", BASE)
